In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu

In [25]:
train_beneficiary = pd.read_csv(
    r"C:\Users\sneha\Downloads\Medical Provider Fraud Detection\Train_Beneficiarydata.csv"
)

train_inpatient = pd.read_csv(
    r"C:\Users\sneha\Downloads\Medical Provider Fraud Detection\Train_Inpatientdata.csv"
)

train_outpatient = pd.read_csv(
    r"C:\Users\sneha\Downloads\Medical Provider Fraud Detection\Train_Outpatientdata.csv"
)

train_provider = pd.read_csv(
    r"C:\Users\sneha\Downloads\Medical Provider Fraud Detection\TRAIN.csv"
)

In [28]:
print("Beneficiary:", train_beneficiary.shape)
print("Inpatient:", train_inpatient.shape)
print("Outpatient:", train_outpatient.shape)
print("Provider:", train_provider.shape)

Beneficiary: (138556, 25)
Inpatient: (40474, 31)
Outpatient: (517737, 27)
Provider: (5410, 2)


In [29]:
print("Train Provider Columns:")
print(train_provider.columns.tolist())

Train Provider Columns:
['Provider', 'PotentialFraud']


In [30]:
train_provider["PotentialFraud"].value_counts(dropna=False)

PotentialFraud
No     4904
Yes     506
Name: count, dtype: int64

In [31]:
train_provider["PotentialFraud"].value_counts(normalize=True).mul(100).round(2)

PotentialFraud
No     90.65
Yes     9.35
Name: proportion, dtype: float64

In [32]:
print(train_provider.columns.tolist())

['Provider', 'PotentialFraud']


In [33]:
print(train_inpatient.columns.tolist())

['BeneID', 'ClaimID', 'ClaimStartDt', 'ClaimEndDt', 'Provider', 'InscClaimAmtReimbursed', 'AttendingPhysician', 'OperatingPhysician', 'OtherPhysician', 'AdmissionDt', 'ClmAdmitDiagnosisCode', 'DeductibleAmtPaid', 'DischargeDt', 'DiagnosisGroupCode', 'ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3', 'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6', 'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9', 'ClmDiagnosisCode_10', 'ClmProcedureCode_1', 'ClmProcedureCode_2', 'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5', 'ClmProcedureCode_6', 'length_of_stay']


In [34]:
print(
    train_provider["PotentialFraud"]
    .value_counts(dropna=False)
)

PotentialFraud
No     4904
Yes     506
Name: count, dtype: int64


# Prepare Inpatient Data

In [35]:
date_columns = [
    "claimstartdt",
    "claimenddt",
    "admissiondt",
    "dischargedt"
]

for col in date_columns:
    if col in train_inpatient.columns:
        train_inpatient[col] = pd.to_datetime(
            train_inpatient[col],
            errors="coerce"
        )

# Create Length of Stay

In [36]:
train_inpatient["AdmissionDt"] = pd.to_datetime(
    train_inpatient["AdmissionDt"],
    errors="coerce"
)

train_inpatient["DischargeDt"] = pd.to_datetime(
    train_inpatient["DischargeDt"],
    errors="coerce"
)

In [37]:
train_inpatient["length_of_stay"] = (
    train_inpatient["DischargeDt"]
    - train_inpatient["AdmissionDt"]
).dt.days

In [38]:
train_inpatient["length_of_stay"].describe()

count    40474.000000
mean         5.665168
std          5.638538
min          0.000000
25%          2.000000
50%          4.000000
75%          7.000000
max         35.000000
Name: length_of_stay, dtype: float64

In [39]:
display(
    train_inpatient[
        [
            "AdmissionDt",
            "DischargeDt",
            "length_of_stay"
        ]
    ].head(10)
)

,AdmissionDt,DischargeDt,length_of_stay
0,2009-04-12,2009-04-18,6
1,2009-08-31,2009-09-02,2
2,2009-09-17,2009-09-20,3
3,2009-02-14,2009-02-22,8
4,2009-08-13,2009-08-30,17
5,2009-10-06,2009-10-12,6
6,2009-01-02,2009-01-07,5
7,2009-08-03,2009-08-07,4
8,2009-08-06,2009-08-09,3
9,2008-12-29,2009-01-05,7


In [40]:
invalid_stays = (
    train_inpatient["length_of_stay"] < 0
).sum()

print("Invalid stays:", invalid_stays)

Invalid stays: 0


In [41]:
print(
    "Missing AdmissionDt:",
    train_inpatient["AdmissionDt"].isna().sum()
)

print(
    "Missing DischargeDt:",
    train_inpatient["DischargeDt"].isna().sum()
)

Missing AdmissionDt: 0
Missing DischargeDt: 0


# High-Cost Inpatient Claim

In [45]:
ip_high_cost_threshold = (
    train_inpatient["InscClaimAmtReimbursed"]
    .quantile(0.90)
)

print(
    "Inpatient High-Cost Threshold:",
    ip_high_cost_threshold
)

Inpatient High-Cost Threshold: 20000.0


In [46]:
train_inpatient["high_cost_claim"] = (
    train_inpatient["InscClaimAmtReimbursed"]
    >= ip_high_cost_threshold
).astype(int)

# Inpatient Provider-Level Features

In [47]:
train_ip_provider = (
    train_inpatient
    .groupby("Provider")
    .agg(
        inpatient_claims=("ClaimID", "count"),

        inpatient_total_reimbursement=(
            "InscClaimAmtReimbursed",
            "sum"
        ),

        inpatient_avg_reimbursement=(
            "InscClaimAmtReimbursed",
            "mean"
        ),

        inpatient_median_reimbursement=(
            "InscClaimAmtReimbursed",
            "median"
        ),

        inpatient_unique_beneficiaries=(
            "BeneID",
            "nunique"
        ),

        avg_length_of_stay=(
            "length_of_stay",
            "mean"
        ),

        max_length_of_stay=(
            "length_of_stay",
            "max"
        ),

        high_cost_inpatient_claims=(
            "high_cost_claim",
            "sum"
        )
    )
    .reset_index()
)

In [48]:
display(train_ip_provider.head())

,Provider,inpatient_claims,inpatient_total_reimbursement,inpatient_avg_reimbursement,inpatient_median_reimbursement,inpatient_unique_beneficiaries,avg_length_of_stay,max_length_of_stay,high_cost_inpatient_claims
0,PRV51001,5,97000,19400.000000,12000.0,5,5.000000,14,2
1,PRV51003,62,573000,9241.935484,7000.0,53,5.161290,27,3
2,PRV51007,3,19000,6333.333333,6000.0,3,5.333333,7,0
3,PRV51008,2,25000,12500.000000,12500.0,2,4.000000,5,1
4,PRV51011,1,5000,5000.000000,5000.0,1,5.000000,5,0


# Inpatient High-Cost Rate

In [49]:
train_ip_provider["high_cost_inpatient_rate"] = (
    train_ip_provider["high_cost_inpatient_claims"]
    /
    train_ip_provider["inpatient_claims"]
)

# Outpatient High-Cost Claims

In [50]:
op_high_cost_threshold = (
    train_outpatient["InscClaimAmtReimbursed"]
    .quantile(0.90)
)

print(
    "Outpatient High-Cost Threshold:",
    op_high_cost_threshold
)

Outpatient High-Cost Threshold: 700.0


In [51]:
train_outpatient["high_cost_claim"] = (
    train_outpatient["InscClaimAmtReimbursed"]
    >= op_high_cost_threshold
).astype(int)

In [52]:
train_outpatient["high_cost_claim"]

0         0
1         0
2         0
3         0
4         0
         ..
517732    1
517733    0
517734    0
517735    0
517736    0
Name: high_cost_claim, Length: 517737, dtype: int32

# Outpatient Provider Features

In [53]:
train_op_provider = (
    train_outpatient
    .groupby("Provider")
    .agg(
        outpatient_claims=("ClaimID", "count"),

        outpatient_total_reimbursement=(
            "InscClaimAmtReimbursed",
            "sum"
        ),

        outpatient_avg_reimbursement=(
            "InscClaimAmtReimbursed",
            "mean"
        ),

        outpatient_median_reimbursement=(
            "InscClaimAmtReimbursed",
            "median"
        ),

        outpatient_unique_beneficiaries=(
            "BeneID",
            "nunique"
        ),

        high_cost_outpatient_claims=(
            "high_cost_claim",
            "sum"
        )
    )
    .reset_index()
)

# Outpatient High-Cost Rate

In [54]:
train_op_provider["high_cost_outpatient_rate"] = (
    train_op_provider["high_cost_outpatient_claims"]
    /
    train_op_provider["outpatient_claims"]
)

# Combine Inpatient + Outpatient Provider Data

In [55]:
train_provider_features = pd.merge(
    train_ip_provider,
    train_op_provider,
    on="Provider",
    how="outer"
)

In [56]:
train_provider_features = pd.merge(
    train_ip_provider,
    train_op_provider,
    on="Provider",
    how="outer"
)

# Total Claims

In [57]:
train_provider_features["total_claims"] = (
    train_provider_features["inpatient_claims"]
    +
    train_provider_features["outpatient_claims"]
)

In [58]:
train_provider_features["total_claims"]

0        25.0
1       132.0
2        72.0
3        43.0
4        58.0
        ...  
5405      NaN
5406      NaN
5407      NaN
5408      NaN
5409      NaN
Name: total_claims, Length: 5410, dtype: float64

# Total Reimbursement

In [59]:
train_provider_features["total_reimbursement"] = (
    train_provider_features[
        "inpatient_total_reimbursement"
    ]
    +
    train_provider_features[
        "outpatient_total_reimbursement"
    ]
)

# Unique Beneficiaries

In [69]:
all_train_claims = pd.concat(
    [
        train_inpatient[
            ["Provider", "BeneID"]
        ],
        train_outpatient[
            ["Provider", "BeneID"]
        ]
    ],
    ignore_index=True
)

In [70]:
train_beneficiary_counts = (
    all_train_claims
    .groupby("Provider")["BeneID"]
    .nunique()
    .reset_index(
        name="total_unique_beneficiaries"
    )
)

In [71]:
train_beneficiary_counts = (
    all_train_claims
    .groupby("Provider")["BeneID"]
    .nunique()
    .reset_index(
        name="total_unique_beneficiaries"
    )
)

# Average Reimbursement Per Claim

In [72]:
train_provider_features[
    "avg_reimbursement_per_claim"
] = (
    train_provider_features["total_reimbursement"]
    /
    train_provider_features["total_claims"]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

# Unique Beneficiaries

In [74]:
all_train_claims = pd.concat(
    [
        train_inpatient[
            ["Provider", "BeneID"]
        ],
        train_outpatient[
            ["Provider", "BeneID"]
        ]
    ],
    ignore_index=True
)

In [75]:
train_beneficiary_counts = (
    all_train_claims
    .groupby("Provider")["BeneID"]
    .nunique()
    .reset_index(
        name="total_unique_beneficiaries"
    )
)

In [77]:
train_provider_features = pd.merge(
    train_provider_features,
    train_beneficiary_counts,
    on="Provider",
    how="left"
)

In [78]:
train_provider_features

,Provider,inpatient_claims,inpatient_total_reimbursement,inpatient_avg_reimbursement,inpatient_median_reimbursement,inpatient_unique_beneficiaries,avg_length_of_stay,max_length_of_stay,high_cost_inpatient_claims,high_cost_inpatient_rate,...,outpatient_avg_reimbursement,outpatient_median_reimbursement,outpatient_unique_beneficiaries,high_cost_outpatient_claims,high_cost_outpatient_rate,total_claims,total_reimbursement,avg_reimbursement_per_claim,total_unique_beneficiaries_x,total_unique_beneficiaries_y
0,PRV51001,5.0,97000.0,19400.000000,12000.0,5.0,5.000000,14.0,2.0,0.400000,...,382.000000,150.0,19.0,5.0,0.250000,25.0,104640.0,4185.600000,24,24
1,PRV51003,62.0,573000.0,9241.935484,7000.0,53.0,5.161290,27.0,3.0,0.048387,...,466.714286,90.0,66.0,15.0,0.214286,132.0,605670.0,4588.409091,117,117
2,PRV51007,3.0,19000.0,6333.333333,6000.0,3.0,5.333333,7.0,0.0,0.000000,...,213.188406,70.0,56.0,3.0,0.043478,72.0,33710.0,468.194444,58,58
3,PRV51008,2.0,25000.0,12500.000000,12500.0,2.0,4.000000,5.0,1.0,0.500000,...,259.268293,90.0,34.0,5.0,0.121951,43.0,35630.0,828.604651,36,36
4,PRV51011,1.0,5000.0,5000.000000,5000.0,1.0,5.000000,5.0,0.0,0.000000,...,204.035088,70.0,52.0,6.0,0.105263,58.0,16630.0,286.724138,53,53
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5405,PRV57759,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,380.000000,65.0,24.0,4.0,0.142857,NaN,NaN,0.000000,24,24
5406,PRV57760,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,216.818182,85.0,9.0,2.0,0.090909,NaN,NaN,0.000000,9,9
5407,PRV57761,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,225.243902,70.0,67.0,9.0,0.109756,NaN,NaN,0.000000,67,67
5408,PRV57762,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1900.000000,1900.0,1.0,1.0,1.000000,NaN,NaN,0.000000,1,1


# Claims Per Beneficiary

In [87]:
train_provider_features["claims_per_beneficiary"] = (
    train_provider_features["total_claims"]
    /
    train_provider_features["total_unique_beneficiaries"]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

In [80]:
print("Inpatient columns:")
print(train_inpatient.columns.tolist())

print("\nOutpatient columns:")
print(train_outpatient.columns.tolist())

Inpatient columns:
['BeneID', 'ClaimID', 'ClaimStartDt', 'ClaimEndDt', 'Provider', 'InscClaimAmtReimbursed', 'AttendingPhysician', 'OperatingPhysician', 'OtherPhysician', 'AdmissionDt', 'ClmAdmitDiagnosisCode', 'DeductibleAmtPaid', 'DischargeDt', 'DiagnosisGroupCode', 'ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3', 'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6', 'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9', 'ClmDiagnosisCode_10', 'ClmProcedureCode_1', 'ClmProcedureCode_2', 'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5', 'ClmProcedureCode_6', 'length_of_stay', 'high_cost_claim']

Outpatient columns:
['BeneID', 'ClaimID', 'ClaimStartDt', 'ClaimEndDt', 'Provider', 'InscClaimAmtReimbursed', 'AttendingPhysician', 'OperatingPhysician', 'OtherPhysician', 'ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3', 'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6', 'ClmDiagnosisCode_7', 'ClmDiagnosisC

In [81]:
all_train_claims = pd.concat(
    [
        train_inpatient[["Provider", "BeneID"]],
        train_outpatient[["Provider", "BeneID"]]
    ],
    ignore_index=True
)

all_train_claims = all_train_claims.dropna(
    subset=["Provider", "BeneID"]
)

print(all_train_claims.shape)
display(all_train_claims.head())

(558211, 2)


,Provider,BeneID
0,PRV55912,BENE11001
1,PRV55907,BENE11001
2,PRV56046,BENE11001
3,PRV52405,BENE11011
4,PRV56614,BENE11014


In [82]:
all_train_claims = pd.concat(
    [
        train_inpatient[["Provider", "BeneID"]],
        train_outpatient[["Provider", "BeneID"]]
    ],
    ignore_index=True
)

all_train_claims = all_train_claims.dropna(
    subset=["Provider", "BeneID"]
)

print(all_train_claims.shape)
display(all_train_claims.head())

(558211, 2)


,Provider,BeneID
0,PRV55912,BENE11001
1,PRV55907,BENE11001
2,PRV56046,BENE11001
3,PRV52405,BENE11011
4,PRV56614,BENE11014


In [83]:
# Remove old version if it exists
if "total_unique_beneficiaries" in train_provider_features.columns:
    train_provider_features = train_provider_features.drop(
        columns=["total_unique_beneficiaries"]
    )

train_provider_features = train_provider_features.merge(
    train_beneficiary_counts,
    on="Provider",
    how="left"
)

In [84]:
print(
    "total_unique_beneficiaries"
    in train_provider_features.columns
)

True


In [85]:
display(
    train_provider_features[
        [
            "Provider",
            "total_claims",
            "total_unique_beneficiaries"
        ]
    ].head(10)
)

,Provider,total_claims,total_unique_beneficiaries
0,PRV51001,25.0,24
1,PRV51003,132.0,117
2,PRV51007,72.0,58
3,PRV51008,43.0,36
4,PRV51011,58.0,53
5,PRV51021,257.0,208
6,PRV51023,36.0,35
7,PRV51024,39.0,38
8,PRV51025,87.0,76
9,PRV51030,171.0,145


# Reimbursement Per Beneficiary

In [89]:
train_provider_features[
    "reimbursement_per_beneficiary"
] = (
    train_provider_features[
        "total_reimbursement"
    ]
    /
    train_provider_features[
        "total_unique_beneficiaries"
    ]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

# Overall High-Cost Claims

In [90]:
train_provider_features[
    "total_high_cost_claims"
] = (
    train_provider_features[
        "high_cost_inpatient_claims"
    ]
    +
    train_provider_features[
        "high_cost_outpatient_claims"
    ]
)

In [91]:
train_provider_features[
    "overall_high_cost_rate"
] = (
    train_provider_features[
        "total_high_cost_claims"
    ]
    /
    train_provider_features[
        "total_claims"
    ]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

# Inpatient Claim Mix

In [92]:
train_provider_features[
    "inpatient_claim_mix"
] = (
    train_provider_features[
        "inpatient_claims"
    ]
    /
    train_provider_features[
        "total_claims"
    ]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

# Outpatient Claim Mix

In [93]:
train_provider_features[
    "outpatient_claim_mix"
] = (
    train_provider_features[
        "outpatient_claims"
    ]
    /
    train_provider_features[
        "total_claims"
    ]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

# Total Deductible Amount

In [94]:
# Inpatient

ip_deductible = (
    train_inpatient
    .groupby("Provider")["DeductibleAmtPaid"]
    .sum()
    .reset_index(
        name="inpatient_total_deductible"
    )
)

In [95]:
# Outpatient

op_deductible = (
    train_outpatient
    .groupby("Provider")["DeductibleAmtPaid"]
    .sum()
    .reset_index(
        name="outpatient_total_deductible"
    )
)

In [96]:
# Merge 

train_provider_features = pd.merge(
    train_provider_features,
    ip_deductible,
    on="Provider",
    how="left"
)

train_provider_features = pd.merge(
    train_provider_features,
    op_deductible,
    on="Provider",
    how="left"
)

In [97]:
# Fill

train_provider_features[
    [
        "inpatient_total_deductible",
        "outpatient_total_deductible"
    ]
] = train_provider_features[
    [
        "inpatient_total_deductible",
        "outpatient_total_deductible"
    ]
].fillna(0)

In [99]:
# Create total


train_provider_features[
    "total_deductible"
] = (
    train_provider_features[
        "inpatient_total_deductible"
    ]
    +
    train_provider_features[
        "outpatient_total_deductible"
    ]
)

# Diagnosis Complexity

In [100]:
diagnosis_cols_ip = [
    col for col in train_inpatient.columns
    if col.startswith("ClmDiagnosisCode_")
]

In [101]:
train_inpatient["diagnosis_code_count"] = (
    train_inpatient[diagnosis_cols_ip]
    .notna()
    .sum(axis=1)
)

In [102]:
# Aggregate

diagnosis_complexity = (
    train_inpatient
    .groupby("Provider")
    .agg(
        avg_diagnosis_codes=(
            "diagnosis_code_count",
            "mean"
        )
    )
    .reset_index()
)

In [103]:
# Merge

train_provider_features = pd.merge(
    train_provider_features,
    diagnosis_complexity,
    on="Provider",
    how="left"
)

# Procedure Complexity

In [105]:
procedure_cols_ip = [
    col for col in train_inpatient.columns
    if col.startswith("ClmProcedureCode_")
]

In [106]:
train_inpatient["procedure_code_count"] = (
    train_inpatient[procedure_cols_ip]
    .notna()
    .sum(axis=1)
)

In [108]:
# Aggregate

procedure_complexity = (
    train_inpatient
    .groupby("Provider")
    .agg(
        avg_procedure_codes=(
            "procedure_code_count",
            "mean"
        )
    )
    .reset_index()
)

In [109]:
# Merge

train_provider_features = pd.merge(
    train_provider_features,
    procedure_complexity,
    on="Provider",
    how="left"
)

# Merge the Actual Fraud Label

In [110]:
train_provider_features = pd.merge(
    train_provider_features,
    train_provider[
        [
            "Provider",
            "PotentialFraud"
        ]
    ],
    on="Provider",
    how="left"
)

In [111]:
# Check

display(
    train_provider_features[
        [
            "Provider",
            "PotentialFraud",
            "total_claims",
            "total_reimbursement",
            "total_unique_beneficiaries",
            "avg_reimbursement_per_claim",
            "claims_per_beneficiary",
            "reimbursement_per_beneficiary",
            "overall_high_cost_rate",
            "inpatient_claim_mix",
            "outpatient_claim_mix",
            "avg_length_of_stay",
            "avg_diagnosis_codes",
            "avg_procedure_codes"
        ]
    ].head(10)
)

,Provider,PotentialFraud,total_claims,total_reimbursement,total_unique_beneficiaries,avg_reimbursement_per_claim,claims_per_beneficiary,reimbursement_per_beneficiary,overall_high_cost_rate,inpatient_claim_mix,outpatient_claim_mix,avg_length_of_stay,avg_diagnosis_codes,avg_procedure_codes
0,PRV51001,No,25.0,104640.0,24,4185.600000,1.041667,4360.000000,0.280000,0.200000,0.800000,5.000000,7.200000,0.600000
1,PRV51003,Yes,132.0,605670.0,117,4588.409091,1.128205,5176.666667,0.136364,0.469697,0.530303,5.161290,8.112903,0.774194
2,PRV51007,No,72.0,33710.0,58,468.194444,1.241379,581.206897,0.041667,0.041667,0.958333,5.333333,7.333333,0.333333
3,PRV51008,No,43.0,35630.0,36,828.604651,1.194444,989.722222,0.139535,0.046512,0.953488,4.000000,7.500000,1.000000
4,PRV51011,No,58.0,16630.0,53,286.724138,1.094340,313.773585,0.103448,0.017241,0.982759,5.000000,8.000000,0.000000
5,PRV51021,Yes,257.0,348830.0,208,1357.315175,1.235577,1677.067308,0.128405,0.112840,0.887160,4.758621,8.517241,0.758621
6,PRV51023,No,36.0,65610.0,35,1822.500000,1.028571,1874.571429,0.111111,0.250000,0.750000,3.555556,8.000000,0.555556
7,PRV51024,No,39.0,64050.0,38,1642.307692,1.026316,1685.526316,0.000000,0.230769,0.769231,4.333333,7.111111,0.777778
8,PRV51025,No,87.0,118350.0,76,1360.344828,1.144737,1557.236842,0.126437,0.149425,0.850575,7.000000,7.769231,0.923077
9,PRV51030,No,171.0,739760.0,145,4326.081871,1.179310,5101.793103,0.099415,0.479532,0.520468,4.621951,8.097561,0.695122


# Final Feature Dataset Check

In [112]:
print(
    "Final Train Feature Dataset:",
    train_provider_features.shape
)

Final Train Feature Dataset: (5410, 35)


In [113]:
print(
    "Unique Providers:",
    train_provider_features["Provider"].nunique()
)

Unique Providers: 5410


In [114]:
print(
    train_provider_features[
        "PotentialFraud"
    ].value_counts(dropna=False)
)

PotentialFraud
No     4904
Yes     506
Name: count, dtype: int64


In [115]:
# Check missing values

missing_summary = (
    train_provider_features
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_summary[
        missing_summary > 0
    ]
)

total_claims                       3716
total_reimbursement                3716
total_high_cost_claims             3716
inpatient_avg_reimbursement        3318
inpatient_unique_beneficiaries     3318
avg_length_of_stay                 3318
max_length_of_stay                 3318
high_cost_inpatient_claims         3318
high_cost_inpatient_rate           3318
avg_diagnosis_codes                3318
avg_procedure_codes                3318
inpatient_total_reimbursement      3318
inpatient_claims                   3318
inpatient_median_reimbursement     3318
high_cost_outpatient_rate           398
outpatient_unique_beneficiaries     398
high_cost_outpatient_claims         398
outpatient_avg_reimbursement        398
outpatient_total_reimbursement      398
outpatient_claims                   398
outpatient_median_reimbursement     398
dtype: int64

# Save the Feature-Engineered Train Dataset

In [116]:
output_path = (
    folder +
    r"\train_provider_features.csv"
)

train_provider_features.to_csv(
    output_path,
    index=False
)

print("File saved successfully:")
print(output_path)

File saved successfully:
C:\Users\sneha\Downloads\Medical Provider Fraud Detection\train_provider_features.csv
